In [29]:
# Consumption

# 1. Re-project both datasets
# 2. Re-sample both datasets to a parameterised grid size (GRAZING_DISTANCE * 2)
# 3. Get consumption by multiplying the cattle head array by consumption
# 4. Minus the consumption from the availability

import rasterio
import numpy as np

# metres
GRAZING_DISTANCE = 50000
GRID_SIZE = GRAZING_DISTANCE * 2

# tonnes/day * duration
KG_CONSUMPTION_PER_COW_ANNUAL = (11 * 365)/1000

# MJ per day * duration
MJ_CONSUMPTION_PER_COW_ANNUAL = (500 * 365)
MJ_CONSUMPTION_PER_COW_6_MONTH = (50 * 180)
MJ_CONSUMPTION_PER_COW_3_MONTH = (50 * 90)

cattle_file = "../data/raw/GLW/5_Ct_2010_Da.tif"
residue_me_file = "../data/processed/residue_inventory/ruminant_me.tif"

In [21]:
# First fix cattle data
dataset = rasterio.open(cattle_file)
arr = dataset.read(1)
new_arr = np.where(arr==-1.7e+308, 0.0, arr)
dataset.close()

fixed_cattle_dataset = rasterio.open(
    "../data/processed/GLW_no_neg_inf.tif",
    "w",
    driver="GTiff",
    height=dataset.height,
    width=dataset.width,
    count=1,
    dtype=new_arr.dtype,
    crs='+proj=latlong',
    transform=dataset.transform
)
        
fixed_cattle_dataset.write(new_arr, 1)
fixed_cattle_dataset.close()

In [22]:
# 1. Re-project both datasets

import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

def reproject_file(src_file, new_file_name, dst_crs="+proj=cea +lon_0=0 +lat_ts=0 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs"):

    with rasterio.open(src_file) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        with rasterio.open(f'../data/processed/{new_file_name}.tif', 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest)

In [23]:
# 1

reproject_file("../data/processed/GLW_no_neg_inf.tif", "cattle_54034")
reproject_file(residue_me_file, "residues_me_54034")

In [24]:
# 2

import rasterio
from rasterio.enums import Resampling

def resample_raster_file(src_file, upscale_factor=0.1):

    with rasterio.open(src_file) as dataset:

        # resample data to target shape
        data = dataset.read(
            out_shape=(
                dataset.count,
                int(dataset.height * upscale_factor),
                int(dataset.width * upscale_factor)
            ),
            resampling=Resampling.bilinear
        )

        # scale image transform
        transform = dataset.transform * dataset.transform.scale(
            (dataset.width / data.shape[-1]),
            (dataset.height / data.shape[-2])
        )
        
        return data, transform



In [25]:
# 2

reproj_cattle_file = "../data/processed/cattle_54034.tif"
reproj_residue_me_file = "../data/processed/residues_me_54034.tif"

cattle_data, cattle_transform = resample_raster_file(reproj_cattle_file)
residue_data, residue_transform = resample_raster_file(reproj_residue_me_file)

In [34]:
cattle_data = cattle_data[0, :, :]
residue_data = residue_data[0, :, :]

In [35]:
# 3. Get consumption by multiplying the cattle head array by consumption

cattle_consumption_arr_annual = cattle_data * MJ_CONSUMPTION_PER_COW_ANNUAL
cattle_consumption_arr_6_month = cattle_data * MJ_CONSUMPTION_PER_COW_6_MONTH
cattle_consumption_arr_3_month = cattle_data * MJ_CONSUMPTION_PER_COW_3_MONTH

In [36]:
# 4. Minus the consumption from the availability

net_consumption_arr_annual = np.subtract(residue_data, cattle_consumption_arr)
net_consumption_arr_6 = np.subtract(residue_data, cattle_consumption_arr_6_month)
net_consumption_arr_3 = np.subtract(residue_data, cattle_consumption_arr_3_month)

In [37]:
consumption_shortfall_dataset_annual = rasterio.open(
    "../data/processed/consumption_shortfall_annual.tif",
    "w",
    driver="GTiff",
    height=residue_data.shape[0],
    width=residue_data.shape[1],
    count=1,
    dtype=cattle_data.dtype,
    crs="+proj=cea +lon_0=0 +lat_ts=0 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs",
    transform=residue_transform
)

consumption_shortfall_dataset_annual.write(net_consumption_arr_annual, 1)
consumption_shortfall_dataset_annual.close()

consumption_shortfall_dataset_6 = rasterio.open(
    "../data/processed/consumption_shortfall_6_month.tif",
    "w",
    driver="GTiff",
    height=residue_data.shape[0],
    width=residue_data.shape[1],
    count=1,
    dtype=cattle_data.dtype,
    crs="+proj=cea +lon_0=0 +lat_ts=0 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs",
    transform=residue_transform
)

consumption_shortfall_dataset_6.write(net_consumption_arr_6, 1)
consumption_shortfall_dataset_6.close()

consumption_shortfall_dataset_3 = rasterio.open(
    "../data/processed/consumption_shortfall_3_month.tif",
    "w",
    driver="GTiff",
    height=residue_data.shape[0],
    width=residue_data.shape[1],
    count=1,
    dtype=cattle_data.dtype,
    crs="+proj=cea +lon_0=0 +lat_ts=0 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs",
    transform=residue_transform
)

consumption_shortfall_dataset_3.write(net_consumption_arr_3, 1)
consumption_shortfall_dataset_3.close()

In [19]:
# cattle consumption

cattle_dataset = rasterio.open(
    "../data/processed/cattle_energy_consumption_annual.tif",
    "w",
    driver="GTiff",
    height=residue_data.shape[0],
    width=residue_data.shape[1],
    count=1,
    dtype=cattle_data.dtype,
    crs="+proj=cea +lon_0=0 +lat_ts=0 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs",
    transform=residue_transform
)

cattle_dataset.write(cattle_consumption_arr, 1)
cattle_dataset.close()